In [3]:

import torch, transformers, datasets
from tqdm import tqdm

c:\studies-master\text-przetwarzanie\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset = datasets.load_dataset("cornell-movie-review-data/rotten_tomatoes")

In [5]:
dataset = datasets.DatasetDict({
    "train": dataset["train"].shuffle(seed=42).select(range(2000)),
    "validation": dataset["validation"].shuffle(seed=42).select(range(500)),
    "test": dataset["test"].shuffle(seed=42).select(range(1000))
})

In [6]:
for split in ["train", "validation", "test"]:
    print(f"\n=== {split.upper()} SAMPLE ===")
    print(dataset[split][0])


=== TRAIN SAMPLE ===
{'text': '. . . plays like somebody spliced random moments of a chris rock routine into what is otherwise a cliche-riddled but self-serious spy thriller .', 'label': 0}

=== VALIDATION SAMPLE ===
{'text': "the filmmaker's heart is in the right place . . .", 'label': 1}

=== TEST SAMPLE ===
{'text': 'unpretentious , charming , quirky , original', 'label': 1}


In [7]:
pretrained_name = "bert-base-uncased"
tokenizer = transformers.BertTokenizerFast.from_pretrained(pretrained_name)

In [8]:
MAX_LEN = 128
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )
dataset = dataset.map(tokenize_batch, batched=True)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 7148.48 examples/s]


In [9]:
dataset["train"][0]

{'text': '. . . plays like somebody spliced random moments of a chris rock routine into what is otherwise a cliche-riddled but self-serious spy thriller .',
 'label': 0,
 'input_ids': [101,
  1012,
  1012,
  1012,
  3248,
  2066,
  8307,
  11867,
  13231,
  2094,
  6721,
  5312,
  1997,
  1037,
  3782,
  2600,
  9410,
  2046,
  2054,
  2003,
  4728,
  1037,
  18856,
  17322,
  1011,
  21834,
  2094,
  2021,
  2969,
  1011,
  3809,
  8645,
  10874,
  1012,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,

In [10]:
print(tokenizer.decode(101))
print(tokenizer.decode(1))
print(tokenizer.decode(21398))
print(tokenizer.decode(2000))

[CLS]
[unused0]
##working
to


In [11]:
columns = ["input_ids", "attention_mask", "label"]

for split in ["train", "validation", "test"]:
    dataset[split].set_format(type="torch", columns=columns)
dataset["train"][0]

{'label': tensor(0),
 'input_ids': tensor([  101,  1012,  1012,  1012,  3248,  2066,  8307, 11867, 13231,  2094,
          6721,  5312,  1997,  1037,  3782,  2600,  9410,  2046,  2054,  2003,
          4728,  1037, 18856, 17322,  1011, 21834,  2094,  2021,  2969,  1011,
          3809,  8645, 10874,  1012,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,   

In [12]:
train_loader = torch.utils.data.DataLoader(dataset["train"], batch_size=8, shuffle=True)
val_loader = torch.utils.data.DataLoader(dataset["validation"], batch_size=16)
test_loader = torch.utils.data.DataLoader(dataset["test"], batch_size=16)

In [13]:
model = transformers.BertForSequenceClassification.from_pretrained(
    pretrained_name,
    num_labels=2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

cpu


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [15]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
num_warmup_steps = int(0.1 * num_training_steps)
scheduler = transformers.get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

In [16]:
best_val_loss = float("inf")
patience = 2
trigger_times = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{num_epochs}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}:= Average Training Loss: {avg_train_loss:.4f}")

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Validation Epoch {epoch+1}/{num_epochs}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
            val_loss += loss.item()

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total
    print(f"Epoch {epoch+1}:= Average Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        trigger_times = 0
        model.save_pretrained("best_model")
        tokenizer.save_pretrained("best_model")
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print("Early stopping triggered.")
            break

Training Epoch 1/3:  48%|████▊     | 120/250 [25:21<27:28, 12.68s/it]    


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import confusion_matrix

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = sum([p==l for p,l in zip(all_preds, all_labels)]) / len(all_labels)
print(f"Test Accuracy: {acc:.4f}")

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

for i in range(5):
    print(f"Text: {tokenizer.decode(dataset['test'][i]['input_ids'], skip_special_tokens=True)}")
    print(f"Label: {dataset['test'][i]['label']}, Pred: {all_preds[i]}")
    print("-"*50)


In [26]:
loss = outputs.loss
logits = outputs.logits
loss.backward()
optimizer.step()
scheduler.step()
total_loss += loss.item()
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [27]:
example_text = "This movie was really great!"
encoding = tokenizer(
    example_text,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LEN,
    padding="max_length"
    
)

print(encoding)

{'input_ids': tensor([[ 101, 2023, 3185, 2001, 2428, 2307,  999,  102,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0,

In [28]:
input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

print(outputs)
logits = outputs.logits
print(logits)
pred = torch.argmax(logits, dim=-1).item()

print(pred)

SequenceClassifierOutput(loss=None, logits=tensor([[ 0.3976, -0.1875]]), hidden_states=None, attentions=None)
tensor([[ 0.3976, -0.1875]])
0
